# Fantasy Premier League (FPL) Optimizer - Exploratory Data Analysis

This notebook provides a comprehensive analysis of FPL data including:
- Data loading and cleaning
- Statistical analysis of top players
- Distribution analysis
- Correlation analysis
- Feature engineering for predictive modeling

**Author:** FPL Optimizer Pipeline  
**Date:** February 2026

## 1. Import Required Libraries

Import all necessary libraries for data manipulation, analysis, and visualization.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules (in same directory)
from data_ingestion import FPLDataLoader
from data_cleaning import FPLDataCleaner
from feature_engineering import FPLFeatureEngineer

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 2. Task 1: Data Ingestion

Load historical FPL gameweek data for the current and previous seasons.

In [2]:
# Initialize the data loader
loader = FPLDataLoader(base_path='../data')

# Load current and previous season data
df_raw = loader.load_current_and_previous_season()

# Display basic information
print(f"\nDataset Shape: {df_raw.shape}")
print(f"Columns: {len(df_raw.columns)}")
print(f"\nSeasons loaded: {df_raw['season'].unique()}")
print(f"Gameweeks range: GW{df_raw['GW'].min()} to GW{df_raw['GW'].max()}")

Loading recent seasons: ['2024-25', '2025-26']
✓ Loaded 2024-25: 27605 records from merged file
✓ Loaded 2025-26: 18183 records from merged file

Total records loaded: 45,788
Seasons included: ['2024-25', '2025-26']
Gameweeks range: 1 to 38


Dataset Shape: (45788, 54)
Columns: 54

Seasons loaded: ['2024-25' '2025-26']
Gameweeks range: GW1 to GW38


In [3]:
# Preview the raw data
print("Sample of raw data:")
df_raw.head(10)

Sample of raw data:


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,modified,opponent_team,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,starts,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW,season,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles
0,Alex Scott,MID,Bournemouth,1.60,0,0,11,0,12.80,77,0.01,0.01,0.00,1.02,6,1,0,3.60,22.80,2024-08-17T14:00:00Z,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,16,0,0,0,0,1,0,4339,1,1,1,0.00,2,0,0,0,50,False,0,1,2024-25,NaN,NaN,NaN,NaN
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.20,0,0,0,0,0.00,427,0.00,0.00,0.00,0.00,6,0,0,0.00,0.00,2024-08-17T14:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,3,0,0,0,0,1,0,33324,0,1,1,0.00,0,0,0,0,45,True,0,1,2024-25,NaN,NaN,NaN,NaN
2,Tomiyasu Takehiro,DEF,Arsenal,0.00,0,0,0,0,0.00,22,0.00,0.00,0.00,0.00,2,0,0,0.00,0.00,2024-08-17T14:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,20,0,0,0,0,1,0,8462,0,0,2,0.00,0,0,0,0,50,True,0,1,2024-25,NaN,NaN,NaN,NaN
3,Malcolm Ebiowei,MID,Crystal Palace,0.00,0,0,0,0,0.00,197,0.00,0.00,0.00,0.00,8,0,0,0.00,0.00,2024-08-18T13:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,4,0,0,0,0,1,0,716,0,1,2,0.00,0,0,0,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN
4,Ben Brereton Díaz,MID,Southampton,1.00,0,0,-2,0,14.00,584,0.02,0.32,0.30,0.25,5,1,0,3.30,2.60,2024-08-17T14:00:00Z,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,15,0,0,0,0,1,0,66244,1,0,1,16.00,1,0,0,0,55,False,1,1,2024-25,NaN,NaN,NaN,NaN
5,Pau Torres,DEF,Aston Villa,1.90,0,0,17,0,1.90,52,0.01,0.01,0.00,2.46,7,1,0,3.10,29.20,2024-08-17T16:30:00Z,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,19,0,0,0,0,1,0,122710,1,2,1,0.00,2,0,0,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN
6,Joel Ward,DEF,Crystal Palace,1.90,0,0,0,0,0.00,215,0.00,0.00,0.00,0.00,8,0,0,0.00,0.00,2024-08-18T13:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,4,0,0,0,0,1,0,12469,0,1,2,0.00,0,0,0,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN
7,Will Lankshear,FWD,Spurs,1.50,0,0,0,0,0.00,609,0.00,0.00,0.00,0.00,10,0,0,0.00,0.00,2024-08-19T19:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,11,0,0,0,0,1,0,17747,0,1,1,0.00,0,0,0,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN
8,Hwang Hee-chan,MID,Wolves,1.30,0,0,14,0,16.30,550,0.11,0.11,0.00,1.24,2,2,0,2.60,6.00,2024-08-17T14:00:00Z,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1,0,0,0,0,1,0,83019,1,0,2,4.00,2,0,0,0,65,False,0,1,2024-25,NaN,NaN,NaN,NaN
9,Mikey Moore,MID,Spurs,1.50,0,0,0,0,0.00,610,0.00,0.00,0.00,0.00,10,0,0,0.00,0.00,2024-08-19T19:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,11,0,0,0,0,1,0,8413,0,1,1,0.00,0,0,0,0,45,False,0,1,2024-25,NaN,NaN,NaN,NaN


## 3. Task 2: Data Cleaning

Clean the data by handling missing values, converting data types, and encoding categorical variables.

In [4]:
# Check missing values before cleaning
print("Missing values in key columns (before cleaning):")
key_cols = ['total_points', 'minutes', 'was_home', 'opponent_team', 'position', 'team']
missing_before = df_raw[key_cols].isnull().sum()
print(missing_before[missing_before > 0])

Missing values in key columns (before cleaning):
Series([], dtype: int64)


In [5]:
# Initialize cleaner and clean the data
cleaner = FPLDataCleaner()
df_clean = cleaner.clean_data(
    df_raw,
    fill_strategy='smart',
    drop_duplicates=True,
    create_dummies=False  # We'll keep categorical encoding for now
)

Starting data cleaning pipeline...
✓ Removed 384 duplicate rows

Fixing data types...
✓ Data types fixed

Handling missing values (strategy: smart)...


c:\Users\osaze\Desktop\Projects\Technical\Python\Projects\RAG\FPL Optimizer\Fantasy-Premier-League\analysis\data_cleaning.py:221: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df['value'] = df.groupby('name')['value'].fillna(method='ffill')
c:\Users\osaze\Desktop\Projects\Technical\Python\Projects\RAG\FPL Optimizer\Fantasy-Premier-League\analysis\data_cleaning.py:221: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['value'] = df.groupby('name')['value'].fillna(method='ffill')


✓ Missing values handled

Encoding categorical variables...
✓ Categorical variables encoded

CLEANING REPORT
Initial shape: (45788, 54)
Final shape: (45404, 57)
Rows removed: 384
  - Duplicates: 384
  - Invalid rows: 0

Missing values before: 11 columns affected
Missing values after: 11 columns still have missing values
  - mng_clean_sheets: 32351
  - mng_draw: 32351
  - mng_goals_scored: 32351
  - mng_loss: 32351
  - mng_underdog_draw: 32351
  - mng_underdog_win: 32351
  - mng_win: 32351
  - clearances_blocks_interceptions: 27231
  - defensive_contribution: 27231
  - recoveries: 27231
  - tackles: 27231



In [6]:
# Verify data types
print("\nData types of key columns:")
print(df_clean[['total_points', 'minutes', 'was_home', 'opponent_team', 
                'position', 'position_encoded', 'ict_index']].dtypes)


Data types of key columns:
total_points          int64
minutes               int64
was_home               bool
opponent_team         int64
position             object
position_encoded      int64
ict_index           float64
dtype: object


## 4. Task 4: Feature Engineering

Create engineered features including the rolling average `last_3_avg_points`.

In [7]:
# Initialize feature engineer and create features
engineer = FPLFeatureEngineer()
df_features = engineer.create_all_features(df_clean)

Creating engineered features...

Creating rolling average features...
  ✓ Created rolling features for 13 columns, 3 windows
Creating form features...
  ✓ Created form features
Creating per-minute features...
  ✓ Created per-90 features for 7 columns
Creating home/away features...
  ✓ Created home/away features
Creating fixture features...
  ✓ Created fixture features
Creating cumulative features...
  ✓ Created cumulative features for 7 columns
✓ Feature engineering complete!
Total features: 133



In [ ]:
# Display sample with engineered rolling/season average features
print("Sample showing engineered points-average features:")
sample_player = df_features[df_features['name'] == df_features['name'].iloc[100]]

requested_cols = ['name', 'season', 'GW', 'total_points', 'last_3_avg_points',
                  'last_5_avg_points', 'season_avg_points']
display_cols = [col for col in requested_cols if col in sample_player.columns]

sample_player[display_cols].head(15)

Sample showing last_3_avg_points feature:


KeyError: "['last_5_avg_points'] not in index"

## 5. Task 3: Basic EDA & Insights

### 5.1 Top 10 Players by Total Points (2024-25 Season)

In [ ]:
# Filter for the last complete season (2024-25)
df_last_season = df_features[df_features['season'] == '2024-25'].copy()

# Calculate total points per player
top_players = (df_last_season.groupby(['name', 'team', 'position'])
               .agg({
                   'total_points': 'sum',
                   'goals_scored': 'sum',
                   'assists': 'sum',
                   'minutes': 'sum',
                   'GW': 'count'
               })
               .rename(columns={'GW': 'games_played'})
               .sort_values('total_points', ascending=False)
               .head(10))

print("=" * 80)
print("TOP 10 PLAYERS - 2024-25 SEASON")
print("=" * 80)
top_players

In [ ]:
# Visualize top 10 players
fig, ax = plt.subplots(figsize=(12, 6))
top_players_plot = top_players.reset_index()
bars = ax.barh(range(len(top_players_plot)), top_players_plot['total_points'], 
               color=sns.color_palette("rocket", len(top_players_plot)))
ax.set_yticks(range(len(top_players_plot)))
ax.set_yticklabels(top_players_plot['name'])
ax.set_xlabel('Total Points', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Players by Total Points - 2024-25 Season', 
             fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()

# Add value labels on bars
for i, (idx, row) in enumerate(top_players_plot.iterrows()):
    ax.text(row['total_points'] + 2, i, f"{int(row['total_points'])} pts", 
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 Distribution of Total Points per Gameweek

In [ ]:
# Statistical summary of points distribution
print("Statistical Summary of Total Points per Gameweek:")
print("=" * 60)
print(df_features['total_points'].describe())
print("\nPercentiles:")
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    value = np.percentile(df_features['total_points'], p)
    print(f"  {p}th percentile: {value:.2f} points")

In [ ]:
# Create comprehensive distribution plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram with KDE
ax1 = axes[0, 0]
ax1.hist(df_features['total_points'], bins=50, alpha=0.7, color='steelblue', 
         edgecolor='black', density=True)
df_features['total_points'].plot(kind='kde', ax=ax1, color='red', linewidth=2)
ax1.set_xlabel('Total Points', fontsize=11, fontweight='bold')
ax1.set_ylabel('Density', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Total Points per Gameweek (with KDE)', 
              fontsize=12, fontweight='bold')
ax1.axvline(df_features['total_points'].mean(), color='green', 
            linestyle='--', linewidth=2, label=f"Mean: {df_features['total_points'].mean():.2f}")
ax1.axvline(df_features['total_points'].median(), color='orange', 
            linestyle='--', linewidth=2, label=f"Median: {df_features['total_points'].median():.2f}")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Box plot
ax2 = axes[0, 1]
bp = ax2.boxplot(df_features['total_points'], vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7),
                 medianprops=dict(color='red', linewidth=2),
                 whiskerprops=dict(linewidth=1.5),
                 capprops=dict(linewidth=1.5))
ax2.set_ylabel('Total Points', fontsize=11, fontweight='bold')
ax2.set_title('Box Plot of Total Points', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Distribution by Position
ax3 = axes[1, 0]
positions = ['GK', 'DEF', 'MID', 'FWD']
position_data = [df_features[df_features['position'] == pos]['total_points'].values 
                 for pos in positions]
bp_pos = ax3.boxplot(position_data, labels=positions, patch_artist=True,
                     boxprops=dict(alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
# Color each position differently
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
for patch, color in zip(bp_pos['boxes'], colors):
    patch.set_facecolor(color)
ax3.set_xlabel('Position', fontsize=11, fontweight='bold')
ax3.set_ylabel('Total Points', fontsize=11, fontweight='bold')
ax3.set_title('Points Distribution by Position', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# 4. Points distribution: Home vs Away
ax4 = axes[1, 1]
home_points = df_features[df_features['was_home'] == True]['total_points']
away_points = df_features[df_features['was_home'] == False]['total_points']
ax4.hist([home_points, away_points], bins=30, alpha=0.6, 
         label=['Home', 'Away'], color=['green', 'blue'], edgecolor='black')
ax4.set_xlabel('Total Points', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax4.set_title('Points Distribution: Home vs Away', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

# Add stats
home_avg = home_points.mean()
away_avg = away_points.mean()
ax4.axvline(home_avg, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax4.axvline(away_avg, color='blue', linestyle='--', linewidth=2, alpha=0.7)
ax4.text(0.98, 0.95, f'Home avg: {home_avg:.2f}\nAway avg: {away_avg:.2f}', 
         transform=ax4.transAxes, fontsize=10, verticalalignment='top',
         horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### 5.3 Correlation Analysis

Identify which features are most strongly correlated with `total_points`.

In [ ]:
# Select features for correlation analysis
correlation_features = [
    'total_points', 'ict_index', 'influence', 'creativity', 'threat',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'minutes', 'bps', 'bonus', 'goals_scored', 'assists',
    'clean_sheets', 'selected', 'value'
]

# Filter features that exist in the dataframe
available_features = [f for f in correlation_features if f in df_features.columns]

# Calculate correlation matrix
correlation_matrix = df_features[available_features].corr()

# Get correlations with total_points and sort
correlations_with_points = correlation_matrix['total_points'].sort_values(ascending=False)
print("Correlations with Total Points:")
print("=" * 60)
print(correlations_with_points)

In [ ]:
# Create correlation heatmap
fig, ax = plt.subplots(figsize=(14, 12))

# Create heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax)

ax.set_title('Correlation Matrix: FPL Features vs Total Points', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize top correlations with total_points
top_correlations = correlations_with_points.drop('total_points').head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in top_correlations.values]
bars = ax.barh(range(len(top_correlations)), top_correlations.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(top_correlations)))
ax.set_yticklabels(top_correlations.index)
ax.set_xlabel('Correlation Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Top 10 Features Correlated with Total Points', 
             fontsize=12, fontweight='bold', pad=15)
ax.axvline(0, color='black', linewidth=0.8, linestyle='-')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(top_correlations.values):
    ax.text(v + 0.01 if v > 0 else v - 0.01, i, f'{v:.3f}', 
            va='center', ha='left' if v > 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Additional Insights

### 6.1 Top Goal Scorers and Assisters

In [ ]:
# Top goal scorers
top_scorers = (df_last_season.groupby(['name', 'position', 'team'])['goals_scored']
               .sum()
               .sort_values(ascending=False)
               .head(10))

# Top assisters
top_assisters = (df_last_season.groupby(['name', 'position', 'team'])['assists']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10))

# Create side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top scorers
scorers_df = top_scorers.reset_index()
ax1.barh(range(len(scorers_df)), scorers_df['goals_scored'], 
         color=sns.color_palette("Reds_r", len(scorers_df)))
ax1.set_yticks(range(len(scorers_df)))
ax1.set_yticklabels(scorers_df['name'])
ax1.set_xlabel('Goals Scored', fontsize=11, fontweight='bold')
ax1.set_title('Top 10 Goal Scorers - 2024-25', fontsize=12, fontweight='bold')
ax1.invert_yaxis()
for i, v in enumerate(scorers_df['goals_scored']):
    ax1.text(v + 0.2, i, str(int(v)), va='center', fontweight='bold')

# Top assisters
assisters_df = top_assisters.reset_index()
ax2.barh(range(len(assisters_df)), assisters_df['assists'], 
         color=sns.color_palette("Blues_r", len(assisters_df)))
ax2.set_yticks(range(len(assisters_df)))
ax2.set_yticklabels(assisters_df['name'])
ax2.set_xlabel('Assists', fontsize=11, fontweight='bold')
ax2.set_title('Top 10 Assist Providers - 2024-25', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
for i, v in enumerate(assisters_df['assists']):
    ax2.text(v + 0.2, i, str(int(v)), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 6.2 Value Analysis - Points per Million

In [ ]:
# Calculate points per million (value metric)
value_analysis = (df_last_season.groupby(['name', 'position', 'team'])
                  .agg({
                      'total_points': 'sum',
                      'value': 'mean',  # Average price
                      'minutes': 'sum',
                      'GW': 'count'
                  })
                  .rename(columns={'GW': 'games_played'}))

# Filter players with significant playing time
value_analysis = value_analysis[value_analysis['minutes'] > 500]  # At least 500 minutes

# Calculate points per million
value_analysis['points_per_million'] = value_analysis['total_points'] / (value_analysis['value'] / 10)

# Top value players
top_value = value_analysis.sort_values('points_per_million', ascending=False).head(15)

print("Top 15 Value Players (min. 500 minutes played):")
print("=" * 80)
top_value[['total_points', 'value', 'points_per_million', 'games_played']]

In [ ]:
# Scatter plot: Value vs Total Points
fig, ax = plt.subplots(figsize=(12, 7))

# Create scatter plot colored by position
positions = df_last_season['position'].unique()
colors_map = {'GK': '#ff9999', 'DEF': '#66b3ff', 'MID': '#99ff99', 'FWD': '#ffcc99'}

for pos in positions:
    pos_data = value_analysis.reset_index()
    pos_data = pos_data[pos_data['position'] == pos]
    ax.scatter(pos_data['value'] / 10, pos_data['total_points'], 
              label=pos, alpha=0.6, s=100, color=colors_map.get(pos, 'gray'))

ax.set_xlabel('Player Value (£m)', fontsize=11, fontweight='bold')
ax.set_ylabel('Total Points', fontsize=11, fontweight='bold')
ax.set_title('Player Value vs Total Points - 2024-25 Season', 
             fontsize=13, fontweight='bold', pad=15)
ax.legend(title='Position', fontsize=10)
ax.grid(True, alpha=0.3)

# Add trend line
from numpy.polynomial import Polynomial
value_array = (value_analysis['value'] / 10).values
points_array = value_analysis['total_points'].values
p = Polynomial.fit(value_array, points_array, 1)
x_trend = np.linspace(value_array.min(), value_array.max(), 100)
y_trend = p(x_trend)
ax.plot(x_trend, y_trend, 'r--', linewidth=2, alpha=0.7, label='Trend')

plt.tight_layout()
plt.show()

### 6.3 Form Analysis - Recent Performance Trends

In [ ]:
# Analyze form using our engineered features
# Get top players and their recent form
recent_gw = df_last_season['GW'].max()
recent_form = df_last_season[df_last_season['GW'] >= recent_gw - 5].copy()

form_analysis = (recent_form.groupby(['name', 'position', 'team'])
                 .agg({
                     'total_points': 'sum',
                     'minutes': 'sum',
                     'GW': 'count'
                 })
                 .rename(columns={'GW': 'games_in_last_5', 'total_points': 'points_last_5'})
                 .sort_values('points_last_5', ascending=False))

# Filter players who played at least 3 games in last 5 GWs
in_form = form_analysis[form_analysis['games_in_last_5'] >= 3].head(15)

print(f"Players in Best Form (Last 5 Gameweeks - GW{recent_gw-4} to GW{recent_gw}):")
print("=" * 80)
in_form

In [ ]:
# Compare form (last 5 GWs) vs overall season average
# Select top 10 overall players
top_10_names = top_players.reset_index()['name'].head(10).tolist()

# Get their data
top_10_data = df_last_season[df_last_season['name'].isin(top_10_names)].copy()

# Calculate season average and recent form
comparison_data = []
for player in top_10_names:
    player_data = top_10_data[top_10_data['name'] == player]
    season_avg = player_data['total_points'].mean()
    recent_avg = player_data[player_data['GW'] >= recent_gw - 5]['total_points'].mean()
    
    comparison_data.append({
        'Player': player,
        'Season Avg': season_avg,
        'Last 5 GW Avg': recent_avg,
        'Form Difference': recent_avg - season_avg
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Form Difference', ascending=False)

# Visualize form comparison
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Season Avg'], width, 
               label='Season Average', alpha=0.8, color='skyblue')
bars2 = ax.bar(x + width/2, comparison_df['Last 5 GW Avg'], width, 
               label='Last 5 GW Average', alpha=0.8, color='orange')

ax.set_xlabel('Player', fontsize=11, fontweight='bold')
ax.set_ylabel('Average Points per GW', fontsize=11, fontweight='bold')
ax.set_title('Form Analysis: Season Average vs Recent Form (Top 10 Players)', 
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Player'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nForm Comparison Table:")
comparison_df

## 7. Summary & Key Findings

### Key Insights from the Analysis:

1. **Data Quality**: Successfully loaded and cleaned data from 2 seasons with comprehensive handling of missing values

2. **Top Performers**: Identified the highest-scoring players based on total points

3. **Point Distribution**:
   - Most gameweeks see players scoring between 0-6 points
   - High-scoring gameweeks (10+ points) are relatively rare but impactful
   - Midfielders and Forwards tend to have higher variance in scoring

4. **Feature Correlations**:
   - **Strongest predictors** of total_points: bonus, goals_scored, assists, bps, ict_index
   - **ICT components** (influence, creativity, threat) all show strong positive correlations
   - **Expected stats** (xG, xA) are good leading indicators

5. **Value Analysis**:
   - Some mid-priced players offer excellent points-per-million value
   - Price doesn't always correlate linearly with points

6. **Form Trends**:
   - Recent form (last 3-5 gameweeks) is a critical predictor through the `last_3_avg_points` feature
   - Player form can fluctuate significantly from season averages

### Next Steps for FPL Optimizer:
1. Build predictive models using the engineered features
2. Develop optimization algorithms for team selection
3. Create fixture difficulty analysis
4. Implement transfer strategy recommendations
5. Build captain selection model

## 8. Export Processed Data

Save the cleaned and feature-engineered dataset for modeling.

In [ ]:
# Save the processed data
output_path = '../data/processed_fpl_data.csv'
df_features.to_csv(output_path, index=False)
print(f"✓ Processed data saved to: {output_path}")
print(f"  Shape: {df_features.shape}")
print(f"  Features: {len(df_features.columns)}")